# 年轮 growth_ring · 宣传视频渲染

本地写好 Remotion 代码 → `git push` → 在这个 notebook 里逐格运行 → 下载 MP4。

**流程**：① 环境 → ② 拉代码 → ③ 装依赖（带缓存）→ ④ 快速预览 → ⑤ 正式渲染 / ⑤b 批量渲染 → ⑥ 下载。

> 免费 Colab 用 CPU 渲染 1080×1920 / 45 秒，通常需要 10～30 分钟。建议先跑第 ④ 步预览。
>
> **省时间**：node_modules 会缓存到你的 Google Drive，第二次运行起跳过安装；批量渲染一次出多个变体。

In [ ]:
# ① 检查环境：Node 版本（Remotion 需要 Node >= 16，这里按 >=18 检查）
import subprocess, re, os

def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"命令失败（退出码 {r.returncode}）: {cmd}")
    return r

node_ver = subprocess.run("node -v", shell=True, capture_output=True, text=True).stdout.strip()
print("当前 Node:", node_ver or "未安装")
m = re.search(r"v(\d+)", node_ver)
major = int(m.group(1)) if m else 0
if major < 18:
    print("Node 版本过低，安装 Node 22 ...")
    run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash -")
    run("apt-get install -y nodejs")
else:
    print("Node 版本满足要求。")
run("node -v && npm -v")

# 按实际 CPU 核数决定并发，避免核少时开太多 Chrome 进程反而变慢
CPU_COUNT = min(os.cpu_count() or 2, 6)
print(f"检测到 {os.cpu_count()} 核，渲染并发设为 {CPU_COUNT}")

In [ ]:
# ② 拉取代码
# ★ 首次使用请把 REPO_URL 改成你的新仓库地址（本项目是独立仓库，不是 growth_ring 的子目录）。
# 公开仓库直接用地址；私有仓库换成：
#   https://<用户名>:<PersonalAccessToken>@github.com/<用户名>/<仓库名>.git
import os, shutil

REPO_URL = "https://github.com/frank-2312/growth_ring_promo.git"  # ← 改成你的实际仓库
BRANCH   = "main"
REPO_DIR = "/content/growth_ring_promo"
PROJ     = REPO_DIR  # 新仓库根目录就是 Remotion 项目本身

if os.path.isdir(f"{REPO_DIR}/.git"):
    print("仓库已存在，拉取最新代码 ...")
    run(f"cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}")
else:
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    run(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}")

assert os.path.isdir(f"{PROJ}/src"), "没有找到项目文件，检查仓库地址/分支是否正确"
os.chdir(PROJ)
print("工作目录:", os.getcwd())
print("字体文件:", os.path.exists(f"{PROJ}/public/fonts/LXGWWenKai-Regular.ttf"))

### ③ 挂载 Drive 并缓存依赖

第一次会把 `node_modules` 打包传到你的 Google Drive（约几十 MB），以后重跑直接解压，跳过 npm install。
如果改了 `package.json`（比如升级 Remotion），把 `USE_CACHE` 改成 `False` 重装一次。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

USE_CACHE = True
CACHE_ZIP = "/content/drive/MyDrive/growth_ring_node_modules.tar.gz"
NODE_MODULES = f"{PROJ}/node_modules"

os.chdir(PROJ)
if USE_CACHE and os.path.exists(CACHE_ZIP):
    print("从 Drive 恢复 node_modules ...")
    if os.path.exists(NODE_MODULES):
        shutil.rmtree(NODE_MODULES)
    run(f"tar -xzf {CACHE_ZIP} -C {PROJ}")
    print("已恢复。若依赖有变动，请把 USE_CACHE 改成 False 重跑。")
else:
    print("安装依赖并缓存到 Drive ...")
    run("npm install --no-audit --no-fund")
    run(f"tar -czf {CACHE_ZIP} -C {PROJ} node_modules")
    print("已缓存到 Drive，下次运行可跳过安装。")

### ④ 快速预览（推荐先跑）

只渲染前 4 秒、半分辨率，用来确认字体、布局、动画正常。没问题再跑正式渲染。
想验证别的片段，改 `--frames` 的范围（如 `--frames=480-600` 看「懂」那段）。

In [ ]:
import time
os.chdir(PROJ)
t0 = time.time()
run(f"cd {PROJ} && npx remotion render src/index.ts GrowthRingPromo out/preview.mp4 "
    f"--codec=h264 --scale=0.5 --frames=0-119 --concurrency={CPU_COUNT} --log=error")
print(f"预览渲染完成，用时 {time.time()-t0:.0f} 秒")

from google.colab import files
files.download(f"{PROJ}/out/preview.mp4")

### ⑤ 正式渲染（单个变体）

全片 1080×1920 / 45 秒。免费 Colab 上请耐心等待，运行时不要断开。

In [ ]:
os.chdir(PROJ)
t0 = time.time()
run(f"cd {PROJ} && npx remotion render src/index.ts GrowthRingPromo out/growth-ring-promo.mp4 "
    f"--codec=h264 --concurrency={CPU_COUNT} --log=error")
print(f"正式渲染完成，用时 {time.time()-t0:.0f} 秒")

### ⑤b 批量渲染（一次出多个变体）

想一次产出多个版本（不同文案 / 配色），在 `variants/` 下加 JSON 配置即可，然后跑这里。
脚本会把每个变体与 `default.json` 深合并 —— 变体里只需写要覆盖的字段。

- `node render-batch.mjs`：渲染全部变体
- `node render-batch.mjs moss privacy`：只渲染指定变体
- 加 `--scale=0.5` 或 `--frames=0-119` 可快速出样片

渲染完会把 `out/` 下的 mp4 打包下载。

In [ ]:
os.chdir(PROJ)
t0 = time.time()
# 只渲染指定变体就改这里，例如：node render-batch.mjs moss privacy
run(f"cd {PROJ} && node render-batch.mjs --concurrency={CPU_COUNT}")
print(f"批量渲染完成，用时 {time.time()-t0:.0f} 秒")

# 打包所有变体成一个 zip 下载
import glob, zipfile
mp4s = [f for f in glob.glob(f"{PROJ}/out/*.mp4") if not os.path.basename(f).startswith('preview')]
zip_path = f"{PROJ}/out/all-variants.zip"
with zipfile.ZipFile(zip_path, 'w') as z:
    for f in mp4s:
        z.write(f, os.path.basename(f))
print(f"打包 {len(mp4s)} 个视频：{[os.path.basename(f) for f in mp4s]}")
files.download(zip_path)

In [ ]:
# ⑥ 下载单个成片（如果跑的是⑤正式渲染）
out = f"{PROJ}/out/growth-ring-promo.mp4"
if os.path.exists(out):
    print(f"文件大小: {os.path.getsize(out)/1024/1024:.1f} MB")
    files.download(out)
else:
    print("没有找到单变体成片 —— 如果你跑的是⑤b批量渲染，去上一个 cell 下载 zip。")

---

### 常见问题

- **npm install 卡住**：重跑③，并把 `USE_CACHE` 设 `False`。
- **Chrome 下载失败**：首次渲染会下载 Chrome Headless Shell（约 150MB），需访问 Google 存储；受限网络下先跑 `npx remotion browser ensure`。
- **字体变方块**：确认②里字体文件存在；`public/fonts/` 的 ttf 随仓库同步，本地改了忘 push 就会出现。
- **改了文案没生效**：默认文案在 `src/content.ts`，变体覆盖在 `variants/*.json`；改完要 push，再重跑②。
- **缓存的依赖版本不对**：改了 `package.json` 后，把 `USE_CACHE=False` 跑一次③重建缓存。
- **渲染太慢 / 运行时快断**：批量渲染一次跑多个变体容易超时，建议分批（指定变体名）或用 `--scale=0.75`。
- **所有 @remotion/* 版本必须一致**：升级时不要只升一个包。